In [5]:
import os
import re
from collections import defaultdict

# Define the question groups and options
question_map = {
    "Q1": ["Yes", "Partially", "No"],
    "Q2": ["Yes", "Partially", "No"],
    "Q3": ["Yes", "Partially", "No"],
    "Q4": ["Yes", "Partially", "No"],
    "Q5": ["Yes", "Partially", "No"],
    "Q6": ["Yes", "Partially", "No"],
    "Q7": ["More Confident", "No change", "Less confident"],
    "Q8": ["Yes", "Maybe", "No"],
    "Q9": ["M1", "M2", "M3", "M4"],
    "Q10": ["M1", "M2", "M3", "M4"]
}

response_counts = {q: defaultdict(int) for q in question_map}

def extract_answer(line):
    match = re.search(r'\[(x|X)\]\s*(.*?)$', line)
    return match.group(2).strip() if match else None

def parse_survey(text):
    lines = text.splitlines()
    q_number = 1
    for line in lines:
        line = line.strip()
        if q_number <= 8:
            answer = extract_answer(line)
            if answer in question_map[f"Q{q_number}"]:
                response_counts[f"Q{q_number}"][answer] += 1
                q_number += 1
        elif "most helpful" in line.lower():
            q_number = 9
        elif "least helpful" in line.lower():
            q_number = 10
        elif q_number in [9, 10]:
            if '[x]' in line.lower():
                match = re.search(r'\[x\]\s*(M\d)', line, re.IGNORECASE)
                if match:
                    method = match.group(1).upper()
                    response_counts[f"Q{q_number}"][method] += 1
                    q_number += 1

def format_percent_row(q_key, options):
    total = sum(response_counts[q_key].values())
    return [f"{(response_counts[q_key][opt] / total * 100):.0f}\\%" if total else "0.0\\%" for opt in options]

def generate_latex_table():
    latex = []
    latex.append("\\begin{table}[htb]")
    latex.append("\\caption{Survey results from human evaluators across various question categories. Percentages reflect responses to agreement (Q1–Q6, Q8), confidence change (Q7), and method preference (Q9–Q10).}")
    latex.append("\\label{tab:human_survey}")
    latex.append("\\begin{tabularx}{\\linewidth}{@{}Xccc@{}}")
    latex.append("\\toprule")
    latex.append("\\textbf{Question} & \\textbf{Yes} & \\textbf{Partially} & \\textbf{No} \\\\ \\midrule")

    labels = [
        "Shared meaning", "Label accuracy", "Relevant to decision",
        "Helps understanding", "Easy to understand", "Can explain to others"
    ]

    for i in range(1, 7):
        row = format_percent_row(f"Q{i}", question_map[f"Q{i}"])
        latex.append(f"Q{i} - {labels[i-1]} & {row[0]} & {row[1]} & {row[2]} \\\\")

    latex.append("\\midrule")
    latex.append("\\textbf{Question} & \\textbf{More\\\\Confident} & \\textbf{No Change} & \\textbf{Less\\\\Confident} \\\\ \\midrule")
    row = format_percent_row("Q7", question_map["Q7"])
    latex.append(f"Q7 - Confidence Change & {row[0]} & {row[1]} & {row[2]} \\\\")

    latex.append("\\midrule")
    latex.append("\\textbf{Question} & \\textbf{Yes} & \\textbf{Maybe} & \\textbf{No} \\\\ \\midrule")
    row = format_percent_row("Q8", question_map["Q8"])
    latex.append(f"Q8  - Would Use Again & {row[0]} & {row[1]} & {row[2]} \\\\")

    latex.append("\\midrule")
    latex.append("\\end{tabularx}")
    latex.append("\\begin{tabularx}{\\linewidth}{@{}Xcccc@{}}")
    latex.append("\\textbf{Question} & \\textbf{M1} & \\textbf{M2} & \\textbf{M3} & \\textbf{M4} \\\\ \\midrule")
    row9 = format_percent_row("Q9", question_map["Q9"])
    row10 = format_percent_row("Q10", question_map["Q10"])
    latex.append(f"Q9   -  Most Helpful & {row9[0]} & {row9[1]} & {row9[2]} & {row9[3]} \\\\")
    latex.append(f"Q10  -  Least Helpful & {row10[0]} & {row10[1]} & {row10[2]} & {row10[3]} \\\\")
    latex.append("\\bottomrule")
    latex.append("\\end{tabularx}")
    latex.append("\\end{table}")
    return "\n".join(latex)

def process_all_responses(root_folder):
    for subdir, _, files in os.walk(root_folder):
        for file in files:
            if file.endswith(".txt"):
                path = os.path.join(subdir, file)
                with open(path, 'r', encoding='utf-8') as f:
                    text = f.read()
                    parse_survey(text)
    print(generate_latex_table())

# Example usage:
# python script.py
if __name__ == "__main__":
    process_all_responses("../data/evaluation/cg_results/Imprisonment_serio/")


\begin{table}[htb]
\caption{Survey results from human evaluators across various question categories. Percentages reflect responses to agreement (Q1–Q6, Q8), confidence change (Q7), and method preference (Q9–Q10).}
\label{tab:human_survey}
\begin{tabularx}{\linewidth}{@{}Xccc@{}}
\toprule
\textbf{Question} & \textbf{Yes} & \textbf{Partially} & \textbf{No} \\ \midrule
Q1 - Shared meaning & 58\% & 33\% & 8\% \\
Q2 - Label accuracy & 33\% & 50\% & 17\% \\
Q3 - Relevant to decision & 0\% & 17\% & 83\% \\
Q4 - Helps understanding & 0\% & 8\% & 92\% \\
Q5 - Easy to understand & 0\% & 8\% & 92\% \\
Q6 - Can explain to others & 0\% & 8\% & 92\% \\
\midrule
\textbf{Question} & \textbf{More\\Confident} & \textbf{No Change} & \textbf{Less\\Confident} \\ \midrule
Q7 - Confidence Change & 0\% & 55\% & 45\% \\
\midrule
\textbf{Question} & \textbf{Yes} & \textbf{Maybe} & \textbf{No} \\ \midrule
Q8  - Would Use Again & 0\% & 9\% & 91\% \\
\midrule
\end{tabularx}
\begin{tabularx}{\linewidth}{@{}Xcccc@{}